# Per-cell synapse scores — HT/NALM vs HT/HB

Focused notebook for computing and visualising four per-cell synapse scores
on the blinatumomab MPX dataset. Extracted from
`ht_nalm_vs_ht_hb_analysis.ipynb` to keep the analysis self-contained.

The four scores live on `adata.obs`:

| Score | Cell type | What it captures |
| --- | --- | --- |
| `cd8_synapse`     | CD8 | productive killing synapse (cSMAC + pSMAC + AIM activation) |
| `cd4_synapse`     | CD4 | helper synapse (MHC-II-anchored cluster) |
| `apc_activation`  | B   | productive Signal-2 hub (MHC-II + costim + adhesion) |
| `apc_inhibitory`  | B   | brake-raft (PD-L1/L2, LAIR-1, CD22, PSGL-1, purinergic) |

Each score has two variants:

* **mean** — abundance arm is the mean across panel markers, then z-scored.
* **sum_zscore** — each marker is z-scored individually first, then summed.
  More democratic — every marker contributes equally regardless of variance.

Combined formula in both cases:
```
score = (1/3) · z(abund_arm)  +  (2/3) · z(coloc_arm)
```
Spatial weight is 2× abundance because spatial deployment, not abundance,
is the gate distinguishing productive from frustrated engagement.

Score panels and pair lists live in `b_cell_utils.DEFAULT_SYNAPSE_PANELS`.


In [ ]:
# [0 · Imports & configuration]
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu

sc.set_figure_params(dpi=100, frameon=False)
plt.rcParams['figure.max_open_warning'] = 0

from nalm_utils import *
from b_cell_utils import (
    compute_synapse_scores,
    compute_derived_metrics,
    build_synapse_obs_long,
    plot_synapse_score_grid,
    plot_synapse_ablation_violins,
    build_ablation_table,
    DEFAULT_SYNAPSE_PANELS,
)

CACHE_DIR        = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
ANNOTATED_CACHE  = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'

SYS_HT_NALM = 'NALM-6 + healthy T'       # HT/NALM
SYS_HT_HB   = 'healthy B + healthy T'    # HT/HB

In [ ]:
# [1 · Data loading]
adata = sc.read_h5ad(ANNOTATED_CACHE)

mask_sys = adata.obs['cell_system'].isin([SYS_HT_NALM, SYS_HT_HB])
print(f'Total cells in HT/NALM + HT/HB: {mask_sys.sum()}')
pd.crosstab(
    index=[adata.obs[mask_sys]['cell_system'], adata.obs[mask_sys]['sample']],
    columns=[adata.obs[mask_sys]['time'], adata.obs[mask_sys]['condition']],
)


## Compute the four scores — both variants

Drop any legacy score columns, then run `compute_synapse_scores` twice:
once with `score_mode='mean'` (writes `cd8_synapse`, etc.) and once with
`score_mode='sum_zscore'` (writes `cd8_synapse_zsum`, etc.).

Derived metrics (`kill_permission`, `apc_functional_state`,
`helper_licensing`, `trogocytosis_score`) are computed from the mean variant.


In [ ]:
# [1.5 · Editable synapse panels — edit markers / pairs here]
# Passed to compute_synapse_scores below as `panels=SYNAPSE_PANELS`.
#
# Keys per panel:
#   cell_types     — cell_type_annot values this score is computed on
#   abundance      — markers averaged for the abundance arm
#   auto_within    — markers whose all-pairs colocs are auto-discovered
#                    (additive, positive)
#   curated_pairs  — explicit cross-cluster pairs added on top (positive)
#   negative_pairs — pairs that SHOULD BE LOW in a productive synapse
#                    (CD45/CD3e exclusion from cSMAC etc.). The coloc
#                    arm becomes  z(agg(pos)) − z(agg(neg)).
SYNAPSE_PANELS = {
    'cd8_synapse': {
        'cell_types': {'CD8'},
        'abundance': [
            'CD3e', 'CD8', 'CD2', 'CD28', 'CD134', 'CD137',
            'CD226', 'TIGIT', 'CD279', 'VISTA',
            'CD11a', 'CD50', 'KLRG1', 'CD94', 'CD48', 'CD352', 'CD53',
            'CD25', 'CD71', 'CD69',
        ],
        'auto_within': [
            'CD3e', 'CD8', 'CD2', 'CD28', 'CD134', 'CD137', 'CD226',
            'TIGIT', 'CD279', 'VISTA',
            'CD11a', 'CD50', 'KLRG1', 'CD94', 'CD48', 'CD352', 'CD53',
        ],
        'curated_pairs': [
            ('CD3e', 'CD8'),
        ],
        'negative_pairs': [
            ('CD3e', 'CD45'), ('CD3e', 'CD43'), ('CD3e', 'CD44'),
            ('CD8',  'CD45'), ('CD8',  'CD43'), ('CD8',  'CD44'),
            ('CD11a','CD45'),
        ],
    },
    'cd4_synapse': {
        'cell_types': {'CD4'},
        'abundance': [
            'CD3e', 'CD4', 'CD2', 'CD28', 'CD134', 'CD137',
            'CD226', 'TIGIT', 'CD279', 'VISTA',
            'CD11a', 'CD50', 'CD48', 'CD352', 'CD53',
            'CD25', 'CD69', 'CD154',
        ],
        'auto_within': [
            'CD3e', 'CD4', 'CD2', 'CD28', 'CD134', 'CD137', 'CD226',
            'TIGIT', 'CD279', 'VISTA',
            'CD11a', 'CD50', 'CD48', 'CD352', 'CD53',
        ],
        'curated_pairs': [
            ('CD3e', 'HLA-DR-DP-DQ'),
            ('CD4',  'HLA-DR-DP-DQ'),
            ('CD2',  'SLAMF6'),
        ],
        'negative_pairs': [
            ('CD3e',         'CD45'), ('CD3e',         'CD43'), ('CD3e',         'CD44'),
            ('CD4',          'CD45'), ('CD4',          'CD43'), ('CD4',          'CD44'),
            ('HLA-DR-DP-DQ', 'CD45'), ('HLA-DR-DP-DQ', 'CD43'), ('HLA-DR-DP-DQ', 'CD44'),
            ('CD11a',        'CD45'),
        ],
    },
    # Naive baseline — abundance + auto coloc over a fixed 17-marker T-cell
    # synapse set, no curated tweaks, no sign surgery. Comparator for the
    # sign-aware cd4_synapse / cd8_synapse panels above.
    'minimal_synapse': {
        'cell_types': {'CD4', 'CD8'},
        'abundance': [
            'CD9', 'CD53', 'CD58', 'CD81',
            'CD2', 'CD3e', 'CD4', 'CD28', 'CD154',
            'CD43', 'CD44', 'CD45',
            'CD11a', 'CD48', 'CD50', 'CD54', 'CD352',
        ],
        'auto_within': [
            'CD9', 'CD53', 'CD58', 'CD81',
            'CD2', 'CD3e', 'CD4', 'CD28', 'CD154',
            'CD43', 'CD44', 'CD45',
            'CD11a', 'CD48', 'CD50', 'CD54', 'CD352',
        ],
        'curated_pairs': [],
        'negative_pairs': [],
    },
    'apc_activation': {
        'cell_types': {'B'},
        'abundance': [
            'HLA-DR-DP-DQ', 'HLA-DR', 'HLA-DQ', 'HLA-ABC',
            'CD80', 'CD86', 'CD40',
            'CD19', 'CD20', 'CD79a',
            'CD54', 'CD58', 'CD50', 'CD102',
        ],
        'auto_within': [
            'HLA-DR-DP-DQ', 'HLA-DR', 'HLA-DQ', 'HLA-ABC',
            'CD80', 'CD86', 'CD40',
            'CD19', 'CD20', 'CD79a',
            'CD54', 'CD58', 'CD50', 'CD102',
        ],
        'curated_pairs': [
            ('CD80', 'HLA-DR-DP-DQ'),
            ('CD86', 'HLA-DR-DP-DQ'),
            ('CD40', 'HLA-DR-DP-DQ'),
            ('CD54', 'HLA-DR-DP-DQ'),
            ('CD58', 'HLA-DR-DP-DQ'),
            ('CD80', 'CD86'),
        ],
        'negative_pairs': [],
    },
    'apc_inhibitory': {
        'cell_types': {'B'},
        'abundance': [
            'CD274', 'CD273', 'CD32', 'CD72', 'CD305',
            'CD22', 'CD66b', 'CD162',
        ],
        'auto_within': [
            'CD274', 'CD273', 'CD32', 'CD72', 'CD305',
            'CD22', 'CD66b', 'CD162',
        ],
        'curated_pairs': [
            ('CD274', 'CD305'),
            ('CD273', 'CD274'),
            ('CD162', 'CD22'),
            ('CD162', 'CD305'),
            ('CD162', 'CD19'),
            ('CD274', 'CD22'),
        ],
        'negative_pairs': [],
    },
}

In [ ]:
# [2 · Compute synapse scores — both variants]
LEGACY_COLS = [
    'synapse_score_immune', 'synapse_score_immune_abundance', 'synapse_score_immune_coloc',
    'synapse_score_apc_productive', 'synapse_score_apc_productive_abundance', 'synapse_score_apc_productive_coloc',
    'synapse_score_apc_inhibitory', 'synapse_score_apc_inhibitory_abundance', 'synapse_score_apc_inhibitory_coloc',
]
adata.obs = adata.obs.drop(columns=[c for c in LEGACY_COLS if c in adata.obs.columns])

print('=== score_mode=mean ===')
scores_mean = compute_synapse_scores(
    adata, panels=SYNAPSE_PANELS,
    layer='arcsinh', coloc_key='spatial_asinh5',
    score_mode='mean',
)
for c in scores_mean.columns:
    adata.obs[c] = scores_mean[c].values

print('\n=== score_mode=sum_zscore ===')
scores_zs = compute_synapse_scores(
    adata, panels=SYNAPSE_PANELS,
    layer='arcsinh', coloc_key='spatial_asinh5',
    score_mode='sum_zscore',
)
for c in scores_zs.columns:
    adata.obs[c + '_zsum'] = scores_zs[c].values

compute_derived_metrics(adata, coloc_key='spatial_asinh5',
                       cd4_baseline_subtract=True)

SCORE_COLS_MEAN = ['cd8_synapse', 'cd4_synapse', 'minimal_synapse',
                   'apc_activation', 'apc_inhibitory']
SCORE_COLS_ZSUM = [c + '_zsum' for c in SCORE_COLS_MEAN]
DERIVED_COLS    = ['kill_permission', 'apc_functional_state',
                   'helper_licensing', 'trogocytosis_score']

print('\n=== mean variant ===')
print(adata.obs[SCORE_COLS_MEAN].describe().round(2))

print('\n=== sum_zscore variant ===')
print(adata.obs[SCORE_COLS_ZSUM].describe().round(2))

print('\n=== cross-variant correlation (per cell type) ===')
for sname in SCORE_COLS_MEAN:
    a, b = adata.obs[sname], adata.obs[sname + '_zsum']
    mask = a.notna() & b.notna()
    r = a[mask].corr(b[mask]) if mask.sum() > 2 else np.nan
    print(f'  {sname:18s}  r = {r:.3f}  (n={mask.sum()})')

## Score distributions — HT/NALM vs HT/HB across time × condition

Single combined figure: **2 rows (variants) × 4 columns (scores)**.
Each subplot is a split violin at four time × condition bins
(`6h Mock` / `6h Blina` / `48h Mock` / `48h Blina`) with the two systems
shown side-by-side via hue (HT/NALM in red, HT/HB in blue). Mann-Whitney
`HT/NALM vs HT/HB` per bin printed below the figure.

The score is z-scored *within its own cell type*, so the y-axis is on
the same scale across systems and time but not directly comparable
between scores.


In [ ]:
# [3 · Combined violin plot — 2 variants × N scores, split by system]
# NOTE: 'minimal_synapse' is computed in [2] but intentionally not plotted.
SCORE_PANELS = [
    ('cd8_synapse',    {'CD8'}),
    ('cd4_synapse',    {'CD4'}),
    ('apc_activation', {'B'}),
    ('apc_inhibitory', {'B'}),
]
VARIANTS    = [('mean', ''), ('sum_zscore', '_zsum')]
TIME_CONDS  = ['6h Mock', '6h Blinatumomab', '48h Mock', '48h Blinatumomab']
SYS_ORDER   = [SYS_HT_NALM, SYS_HT_HB]
SYS_PALETTE = {SYS_HT_NALM: '#d62728', SYS_HT_HB: '#1f77b4'}

obs_v = build_synapse_obs_long(
    adata, SCORE_COLS_MEAN + SCORE_COLS_ZSUM, SYS_ORDER,
)

plot_synapse_score_grid(
    obs_v, SCORE_PANELS, VARIANTS, SYS_ORDER, SYS_PALETTE,
    x_levels=TIME_CONDS, mode='split', col_width=5.0, row_height=3.2,
)
plt.show()

print('=' * 78)
print('HT/NALM vs HT/HB Mann-Whitney per (variant × score × time_cond)')
print('=' * 78)
for variant_name, suffix in VARIANTS:
    for base_col, ctypes in SCORE_PANELS:
        col = base_col + suffix
        sub = obs_v[obs_v['cell_type_annot'].isin(ctypes)]
        for tc in TIME_CONDS:
            nm = sub.loc[(sub['cell_system'] == SYS_HT_NALM) &
                         (sub['time_cond'] == tc), col].dropna().values
            hb = sub.loc[(sub['cell_system'] == SYS_HT_HB) &
                         (sub['time_cond'] == tc), col].dropna().values
            if len(nm) >= 5 and len(hb) >= 5:
                _, p = mannwhitneyu(nm, hb, alternative='two-sided')
                delta = np.median(nm) - np.median(hb)
                print(f'  {variant_name:10s} {base_col:18s} {tc:18s} '
                      f'Δmedian(NALM−HB)={delta:+.3f}  p={p:.2e}')

## Focused 6h comparison — HT/NALM vs HT/HB

Two figures, one per condition at the 6h timepoint
(`6h Blinatumomab`, `6h Mock`). Each figure is **2 rows (variants) ×
4 cols (scores)**; within every subplot the violin compares HT/NALM
against HT/HB. Mann-Whitney p-value annotated above each pair.

In [ ]:
# [4 · 6h-only violins — NALM vs HB, both methods]
FOCUS_TIME_CONDS = ['6h Blinatumomab', '6h Mock']

for tc in FOCUS_TIME_CONDS:
    plot_synapse_score_grid(
        obs_v, SCORE_PANELS, VARIANTS, SYS_ORDER, SYS_PALETTE,
        x_levels=tc, mode='by_system',
        suptitle=f'HT/NALM vs HT/HB — {tc}',
        col_width=3.2, row_height=3.2,
    )
    plt.show()

## Ablation — abundance-only vs spatial-only vs combined

`compute_synapse_scores` writes three columns per panel:

| Suffix | Arm |
| --- | --- |
| `{name}_abundance` | z-scored marker abundance only |
| `{name}_coloc`     | z-scored colocalisation only |
| `{name}`           | combined `1/3 abund + 2/3 coloc` |

Ablation question: at 6h, how much of the HT/NALM vs HT/HB discrimination
comes from each arm? Compares Δmedian and Mann-Whitney `p` for the three
arms across both score variants.

In [ ]:
# [5 · Ablation table — Δmedian + MW p per (variant × score × arm × cond)]
ablation_df = build_ablation_table(
    adata, SCORE_PANELS, VARIANTS, FOCUS_TIME_CONDS, SYS_ORDER,
)

print('=== ablation summary (mean variant, sorted by |d_med|) ===')
print(ablation_df.query('variant == "mean"')
      .assign(abs_d=lambda d: d['d_med'].abs())
      .sort_values('abs_d', ascending=False)
      .drop(columns='abs_d')
      .to_string(index=False, float_format='%.3f'))

In [ ]:
# [6 · Ablation violins — 3 arms × N scores, NALM vs HB at 6h]
# Mean variant only (sum_zscore variant is r > 0.87 with mean — see [2]).
plot_synapse_ablation_violins(
    adata, SCORE_PANELS, FOCUS_TIME_CONDS, SYS_ORDER, SYS_PALETTE,
)
plt.show()